# GIADA Task 0.4 — doppio oracle indipendente per `Ca_HVA`

Questo notebook confronta due percorsi indipendenti: (A) le formule estratte direttamente da `Ca_HVA.mod`; (B) il `DERIVATIVE states` autentico compilato ed eseguito da NEURON. Non addestra alcun modello e non accetta dataset: verifica prima l'identità numerica dei target atomici.

In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys

WORKSPACE = Path('/kaggle/working/giada_task_0_4')
GIADA_REPO = WORKSPACE / 'giada'
TEACHER_REPO = WORKSPACE / 'neuron_as_deep_net'
OUTPUT_DIR = Path('/kaggle/working/artifacts/giada_teacher_double_oracle')
TEACHER_COMMIT = '074c4666300a8ad246601dab179a97a6942f0f29'
GIADA_REF = 'codex/surrogate-validity-audit'
WORKSPACE.mkdir(parents=True, exist_ok=True)


In [ ]:
def run(command, cwd=None):
    print('+', ' '.join(map(str, command)))
    subprocess.run(list(map(str, command)), cwd=cwd, check=True)

if not GIADA_REPO.exists():
    run(['git', 'clone', 'https://github.com/Zagred47/giada.git', GIADA_REPO])
run(['git', 'fetch', 'origin', GIADA_REF], cwd=GIADA_REPO)
run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=GIADA_REPO)
if not TEACHER_REPO.exists():
    run(['git', 'clone', 'https://github.com/SelfishGene/neuron_as_deep_net.git', TEACHER_REPO])
run(['git', 'checkout', '--detach', TEACHER_COMMIT], cwd=TEACHER_REPO)
run([sys.executable, '-m', 'pip', 'install', '-q', 'neuron'])
print({'giada_revision': subprocess.check_output(['git','rev-parse','HEAD'], cwd=GIADA_REPO, text=True).strip(), 'teacher_revision': subprocess.check_output(['git','rev-parse','HEAD'], cwd=TEACHER_REPO, text=True).strip()})


## Esecuzione

La griglia comprende entrambi i gate, l'intervallo -120…60 mV, i punti attorno alla singolarità -27 mV, cinque stati iniziali e quattro passi temporali. La tolleranza preregistrata è `atol=rtol=1e-10` in float64.

In [ ]:
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
OUTPUT_DIR.mkdir(parents=True)
mod_file = TEACHER_REPO / 'L5PC_NEURON_simulation/mods/Ca_HVA.mod'
run([
    sys.executable, GIADA_REPO / 'scripts/run_teacher_double_oracle.py', mod_file,
    '--build-dir', OUTPUT_DIR / 'compiled_mechanism',
    '--json', OUTPUT_DIR / 'teacher_double_oracle_v1.json',
    '--markdown', OUTPUT_DIR / 'teacher_double_oracle_v1.md',
])


In [ ]:
report = json.loads((OUTPUT_DIR / 'teacher_double_oracle_v1.json').read_text())
display({k: report[k] for k in ['valid','failure_count','maximum_absolute_error','grid','tolerance','worst_case']})
assert report['valid'], 'Task 0.4 non superata: non procedere alla generazione del dataset atomico.'
assert report['failure_count'] == 0


## Download dell'artefatto

La cella seguente usa il metodo Blob/base64 compatibile con Kaggle adottato nel progetto.

In [ ]:
from base64 import b64encode
from IPython.display import Javascript, display

zip_base = Path('/kaggle/working/giada_teacher_double_oracle')
zip_path = Path(shutil.make_archive(str(zip_base), 'zip', root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name))
payload = b64encode(zip_path.read_bytes()).decode('ascii')
filename = zip_path.name
display(Javascript(f"""
const binary = atob('{payload}');
const bytes = new Uint8Array(binary.length);
for (let i = 0; i < binary.length; i++) bytes[i] = binary.charCodeAt(i);
const blob = new Blob([bytes], {{type: 'application/zip'}});
const url = URL.createObjectURL(blob);
const a = document.createElement('a');
a.href = url; a.download = '{filename}'; document.body.appendChild(a); a.click(); a.remove();
setTimeout(() => URL.revokeObjectURL(url), 1000);
"""))
print({'zip': str(zip_path), 'size_mib': round(zip_path.stat().st_size / 2**20, 3)})
